# AlexNet 안으로 들어가기: 합성곱, 특징맵, 풀링, 수용영역

이 노트북은 퍼셉트론과 MLP 다음에 CNN이 왜 필요했는지 AlexNet 내부 출력을 직접 보며 이해하는 실습입니다.

학습 순서: 원본 이미지 → 합성곱 특징맵 → 풀링 → 깊이에 따른 수용영역 확대 → MLP와 CNN의 파라미터 구조 비교

> AlexNet은 최초의 CNN이 아니라, 2012년 대규모 이미지 인식에서 딥 CNN의 실용성을 널리 입증한 역사적 전환점입니다.

## 학습 목표

1. 합성곱 필터 하나가 이미지 전체에서 같은 가중치를 공유하는 이유를 설명합니다.
2. 한 입력 이미지가 여러 채널의 특징맵으로 변하는 과정을 관찰합니다.
3. Max pooling과 average pooling이 정보를 요약하는 방식을 비교합니다.
4. 깊은 층의 뉴런이 더 넓은 입력 영역을 보는 이유를 계산합니다.
5. MLP와 CNN의 파라미터 수 차이를 이미지 구조 활용 관점에서 해석합니다.

In [ ]:
import torch
import matplotlib.pyplot as plt
from torchinfo import summary

from cifar10_lab import (
    CIFAR10_MEAN, CIFAR10_STD, create_model, detect_environment,
    load_cifar10_data, resolve_data_dir, set_global_seed,
)
from cifar10_lab.cnn_visualization import (
    collect_feature_maps, plot_feature_map_progression,
    plot_pooling_comparison, plot_receptive_field_progression,
    receptive_field_stages,
)

set_global_seed(42, deterministic=True)
runtime = detect_environment()
print('Runtime:', runtime)

## 1. 한 장의 이미지에서 시작하기

CIFAR-10 이미지는 3개 색상 채널과 32×32 공간으로 이루어집니다. MLP는 이를 3,072개의 숫자로 펼치지만, CNN은 높이와 너비를 유지한 채 작은 필터를 움직입니다. 아래에서는 동일한 한 장을 모든 단계에 사용합니다.

In [ ]:
# 데이터가 없다면 torchvision이 처음 한 번 자동 다운로드합니다.
_, _, testloader, classes = load_cifar10_data(
    batch_size=1, seed=42, num_workers=runtime.num_workers,
    pin_memory=runtime.pin_memory, data_root=resolve_data_dir(),
    max_train_samples=1, max_val_samples=1, max_test_samples=1,
)
image, label = next(iter(testloader))
image = image.to(runtime.device)

# 화면에 표시할 때만 정규화를 되돌립니다. 모델에는 정규화된 입력을 넣습니다.
mean = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
std = torch.tensor(CIFAR10_STD).view(3, 1, 1)
display_image = (image[0].cpu() * std + mean).clamp(0, 1)
plt.figure(figsize=(3.2, 3.2))
plt.imshow(display_image.permute(1, 2, 0))
plt.title(f'Input image: {classes[int(label[0])]}')
plt.axis('off')
plt.show()

## 2. CIFAR-10용 AlexNet 구조

원래 AlexNet은 큰 ImageNet 입력을 사용했습니다. 이 프로젝트의 AlexNet은 핵심 흐름을 유지하면서 32×32 입력에 맞게 첫 합성곱과 pooling 크기를 조정했습니다. 아직 학습하지 않은 모델이므로 다음 특징맵은 **구조와 텐서 변화**를 관찰하는 용도입니다. 학습된 체크포인트를 사용하면 특정 모양에 반응하는 채널이 더 뚜렷해집니다.

In [ ]:
alexnet = create_model('alexnet', num_classes=10, image_size=32).to(runtime.device).eval()
model_summary = summary(alexnet, input_size=(1, 3, 32, 32), depth=3, verbose=0)
print(model_summary)

## 3. 특징맵: 한 이미지를 여러 관점으로 보기

합성곱 층의 각 출력 채널은 서로 다른 필터 결과입니다. 얕은 층은 경계·색 변화 같은 국소 패턴을, 학습된 깊은 층은 더 복합적인 조합을 표현할 수 있습니다. 행이 아래로 갈수록 공간 크기가 줄어드는 것도 확인하세요.

같은 가중치를 모든 위치에 반복 적용하기 때문에 위치마다 별도의 파라미터를 두지 않고도 비슷한 패턴을 여러 위치에서 찾을 수 있습니다.

In [ ]:
# Conv2d와 MaxPool2d를 통과할 때의 중간 출력을 hook으로 기록합니다.
feature_records = collect_feature_maps(alexnet, image, max_layers=6)
for record in feature_records:
    print(record['name'], record['type'], '->', record['shape'])
plot_feature_map_progression(feature_records, max_channels=4)
plt.show()

## 4. Pooling: 작게 만들면서 무엇을 남길 것인가

Max pooling은 작은 영역에서 가장 강한 반응을 남겨 특징의 존재를 강조합니다. Average pooling은 영역의 평균을 남겨 전체적인 경향을 부드럽게 보존합니다. 둘 다 해상도와 계산량을 줄이지만 보존하는 정보가 다릅니다.

In [ ]:
plot_pooling_comparison(image)
plt.show()

## 5. 수용영역: 깊어질수록 더 넓게 보기

수용영역은 특정 특징값 하나가 영향을 받을 수 있는 원본 입력 영역의 크기입니다. 3×3 합성곱도 여러 층을 거치면 이전 층이 본 영역을 다시 조합하므로 더 넓은 문맥을 사용합니다. Pooling의 stride는 다음 층이 입력을 건너뛰는 간격도 키웁니다.

In [ ]:
stages = receptive_field_stages(alexnet)
for stage in stages:
    print(stage)
plot_receptive_field_progression(alexnet)
plt.show()

## 6. 퍼셉트론, MLP, AlexNet, ResNet의 크기 비교

파라미터 수만으로 모델의 우열을 판단할 수는 없습니다. 중요한 질문은 그 파라미터가 이미지 구조를 어떻게 활용하는가입니다. 퍼셉트론은 작지만 선형이고, MLP는 비선형이지만 공간 구조를 펼칩니다. CNN은 필터를 위치 전체에 공유하면서 계층적 특징을 만듭니다.

In [ ]:
model_ids = ('perceptron', 'mlp', 'alexnet', 'resnet18')
parameter_counts = []
for model_id in model_ids:
    model = create_model(model_id, num_classes=10, image_size=32)
    parameter_counts.append(sum(parameter.numel() for parameter in model.parameters()))
figure, axis = plt.subplots(figsize=(8, 4))
bars = axis.bar(model_ids, [value / 1_000_000 for value in parameter_counts])
axis.bar_label(bars, labels=[f'{value / 1_000_000:.2f}M' for value in parameter_counts])
axis.set_ylabel('parameters (millions)')
axis.set_title('Architecture changes matter more than parameter count alone')
figure.tight_layout()
plt.show()

## 다음 실험

1. 명령행에서 AlexNet을 빠르게 학습합니다: cifar10-lab train --model alexnet --quick
2. 학습된 체크포인트를 지정해 그림을 저장합니다: cifar10-lab visualize-cnn --model alexnet --checkpoint-dir 체크포인트폴더
3. ResNet18로 모델을 바꾸고 특징맵과 수용영역 증가 방식을 비교합니다.
4. compare 명령으로 perceptron, mlp, alexnet, resnet18의 정확도·파라미터·추론 시간을 비교합니다.

주의: 랜덤 초기화 특징맵은 텐서 구조를 설명하고, 학습된 특징맵은 데이터에서 배운 반응을 설명합니다. 두 그림의 목적을 구분해야 합니다.